<h2>Create complex interaction features by combining numerical and categorical variables, incorporating domain knowledge to enhance predictive power and evaluating their impact through feature importance analysis.</h2>

In [22]:
import pandas as pd
import numpy as np

In [23]:
from sklearn.ensemble import RandomForestRegressor

In [24]:
np.random.seed(42)
n_samples = 500

In [25]:
df = pd.DataFrame({
    'item_category': np.random.choice(['Electronics', 'Clothing', 'Groceries'], size=n_samples),
    'order_quantity': np.random.randint(1, 10, size=n_samples),
    'unit_price': np.random.uniform(10.0, 500.0, size=n_samples),
    'customer_tenure_months': np.random.randint(1, 48, size=n_samples),
    'shipping_cost': np.random.uniform(5.0, 50.0, size=n_samples)
})

In [26]:
df['gross_order_value'] = df['order_quantity'] * df['unit_price']
category_mean_price = df.groupby('item_category')['unit_price'].transform('mean')
df['price_relative_to_cat_avg'] = df['unit_price'] / category_mean_price
df['shipping_to_order_value_ratio'] = df['shipping_cost'] / (df['gross_order_value'] + 1e-5)
df['loyalty_spend_power'] = df['gross_order_value'] * np.log1p(df['customer_tenure_months'])

In [28]:
target_col = 'total_spend'

In [29]:
if target_col not in df.columns:
    # If using your own dataset, set target_col to your actual target column name
    # e.g., target_col = 'total_amount' 
    if 'total_amount' in df.columns:
        target_col = 'total_amount'
    elif 'selling_price' in df.columns:
        target_col = 'selling_price'
    else:
        # Create total_spend if building from synthetic example
        df['total_spend'] = df['order_quantity'] * df['unit_price'] + df['shipping_cost']

In [30]:
df_model = pd.get_dummies(df, columns=['item_category'], drop_first=True)

In [31]:
X = df_model.drop(columns=[target_col])
y = df_model[target_col]

In [32]:
print("Features (X) and Target (y) created successfully!")

Features (X) and Target (y) created successfully!


In [35]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X, y)

RandomForestRegressor(random_state=42)

In [36]:
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)

In [37]:
print("Feature Importance Ranking:")
print(feature_importance)

Feature Importance Ranking:
                         Feature  Importance
0              gross_order_value    0.999347
1                  shipping_cost    0.000189
2            loyalty_spend_power    0.000132
3      price_relative_to_cat_avg    0.000105
4                     unit_price    0.000077
5  shipping_to_order_value_ratio    0.000072
6         customer_tenure_months    0.000031
7        item_category_Groceries    0.000017
8      item_category_Electronics    0.000016
9                 order_quantity    0.000014
